# 01 — Exploratory Data Analysis
## MTSamples Clinical Notes Dataset

This notebook explores the MTSamples dataset before modeling:
- Dataset overview and null values
- Medical specialty distribution
- Note length distribution
- Text quality after preprocessing
- Simulated label distribution and class balance
- Word clouds by risk class

> **Dataset:** MTSamples — 4,999 de-identified medical transcription samples  
> **Source:** [kaggle.com/datasets/tboyle10/medicaltranscriptions](https://www.kaggle.com/datasets/tboyle10/medicaltranscriptions)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

from src.preprocessing import preprocess_dataframe
from src.label_simulation import generate_labels

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
sns.set_style('whitegrid')
print('Imports OK')

## 1. Load Raw Data

In [ ]:
df_raw = pd.read_csv('../data/mtsamples.csv')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
df_raw.head(3)

In [ ]:
# Null value audit
print('Null values per column:')
print(df_raw.isnull().sum())
print(f'\nRows with null transcription: {df_raw["transcription"].isnull().sum()}')

## 2. Medical Specialty Distribution

In [ ]:
specialty_counts = df_raw['medical_specialty'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
top_20 = specialty_counts.head(20)
colors = ['#e63946' if i < 5 else '#457b9d' for i in range(len(top_20))]
top_20.plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Number of Notes')
ax.set_title('Top 20 Medical Specialties in MTSamples')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/specialty_distribution.png', bbox_inches='tight')
plt.show()

print(f'Total specialties: {df_raw["medical_specialty"].nunique()}')

## 3. Note Length Analysis

In [ ]:
df_raw['word_count'] = df_raw['transcription'].fillna('').apply(lambda x: len(x.split()))
df_raw['char_count'] = df_raw['transcription'].fillna('').apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df_raw['word_count'].clip(upper=2000), bins=50, color='#457b9d', edgecolor='white')
axes[0].axvline(x=400, color='#e63946', linestyle='--', lw=2, label='Truncation limit (400 words)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Note Length Distribution (words)')
axes[0].legend()

axes[1].hist(df_raw['char_count'].clip(upper=10000), bins=50, color='#457b9d', edgecolor='white')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Note Length Distribution (characters)')

plt.tight_layout()
plt.savefig('../results/note_length_distribution.png', bbox_inches='tight')
plt.show()

print(f'Word count stats:')
print(df_raw['word_count'].describe().round(1))
pct_over_400 = (df_raw['word_count'] > 400).mean()
print(f'\nNotes exceeding 400 words (will be truncated): {pct_over_400:.1%}')

## 4. Preprocessing

In [ ]:
df = preprocess_dataframe(df_raw, text_col='transcription')
print(f'After preprocessing: {len(df)} notes (dropped {len(df_raw) - len(df)} empty/short)')

# Show a before/after example
idx = 10
print('\n--- RAW ---')
print(df_raw['transcription'].iloc[idx][:500])
print('\n--- CLEANED ---')
print(df['clean_text'].iloc[idx][:500])

## 5. Simulated Labels

In [ ]:
df = generate_labels(df, text_col='clean_text', target_base_rate=0.20)

print('Label distribution:')
print(df['readmitted_30d'].value_counts())
print(f'\nBase rate: {df["readmitted_30d"].mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label pie chart
label_counts = df['readmitted_30d'].value_counts()
axes[0].pie(
    label_counts.values,
    labels=['Low Risk (0)', 'High Risk (1)'],
    colors=['#457b9d', '#e63946'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[0].set_title('Label Distribution (simulated)')

# Risk score distribution
axes[1].hist(df[df['readmitted_30d'] == 0]['risk_score'], bins=30,
             alpha=0.7, color='#457b9d', label='Low Risk', density=True)
axes[1].hist(df[df['readmitted_30d'] == 1]['risk_score'], bins=30,
             alpha=0.7, color='#e63946', label='High Risk', density=True)
axes[1].set_xlabel('Heuristic Risk Score')
axes[1].set_ylabel('Density')
axes[1].set_title('Risk Score Distribution by Label')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/label_distribution.png', bbox_inches='tight')
plt.show()

## 6. Word Clouds by Risk Class

In [ ]:
from wordcloud import WordCloud, STOPWORDS

# Clinical stopwords to filter out common but non-informative terms
CLINICAL_STOPWORDS = set(STOPWORDS) | {
    'patient', 'history', 'noted', 'reported', 'given', 'also',
    'including', 'well', 'taken', 'performed', 'using', 'show',
    'one', 'two', 'three', 'left', 'right', 'normal', 'within',
    'day', 'week', 'month', 'year', 'time', 'date', 'follow',
    'redacted',
}

high_risk_text = ' '.join(df[df['readmitted_30d'] == 1]['clean_text'])
low_risk_text  = ' '.join(df[df['readmitted_30d'] == 0]['clean_text'])

wc_high = WordCloud(
    width=700, height=350,
    background_color='white',
    colormap='Reds',
    stopwords=CLINICAL_STOPWORDS,
    max_words=80,
).generate(high_risk_text)

wc_low = WordCloud(
    width=700, height=350,
    background_color='white',
    colormap='Blues',
    stopwords=CLINICAL_STOPWORDS,
    max_words=80,
).generate(low_risk_text)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(wc_high, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('High-Risk Notes — Most Frequent Words', fontsize=13, pad=10)

axes[1].imshow(wc_low, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Low-Risk Notes — Most Frequent Words', fontsize=13, pad=10)

plt.tight_layout()
plt.savefig('../results/word_clouds.png', bbox_inches='tight')
plt.show()

## 7. Key Takeaways

- **4,999 notes** across 40 specialties; surgery and internal medicine dominate
- **Median note length: ~300 words** — our 400-word truncation retains most content
- **~20% high-risk** labels (matching real hospital base rates)
- Word clouds confirm that high-risk notes contain disease-heavy language (CHF, COPD, failure, complications) while low-risk notes are more procedural
- This linguistic separation is the signal ClinicalBERT will exploit

→ **Next: `02_Modeling.ipynb`**